# Wan Studio Colab Web UI

Run this notebook when you want Colab to host Wan Studio and generate real Wan videos.

1. Select a GPU runtime in Colab. For real Wan2.2-TI2V-5B generation, use a 24 GB+ GPU such as L4/A100; free T4 can run the UI but will likely fail or run out of memory for real generation.
2. Clone the GitHub repo and install Wan Studio.
3. Mount Google Drive and download or reuse the Wan2.2-TI2V-5B base model plus the optional LoRA set.
4. Install the official Wan runner repository.
5. Start the Web UI in real Wan runner mode and keep the final cell running.


## 1. Clone and install Wan Studio

Run this first. It clones the public GitHub repo into the temporary Colab runtime and installs the app.


In [ ]:
REPO_URL = 'https://github.com/jjj06960-hash/wan-studio.git'
APP_DIR = '/content/wan-studio'

!rm -rf {APP_DIR}
!git clone {REPO_URL} {APP_DIR}
%cd {APP_DIR}
!python install.py --accelerator cuda --system


## 2. Mount Drive and download the real base model and LoRA set once

The LoRA/adapters repo is not enough for generation by itself. For fast starter tests, use Wan2.2 TI2V 5B. For the default A14B I2V LoRA set, set `USE_A14B_I2V_FOR_LORA_PRESET = True` before running this cell.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

USE_A14B_I2V_FOR_LORA_PRESET = False  # Set True for the default A14B I2V LoRA quality preset.

FAST_BASE_REPO_ID = 'Wan-AI/Wan2.2-TI2V-5B'
FAST_BASE_MODEL_DIR = Path('/content/drive/MyDrive/WanStudio/models/Wan2.2-TI2V-5B')
A14B_I2V_REPO_ID = 'Wan-AI/Wan2.2-I2V-A14B'
A14B_I2V_MODEL_DIR = Path('/content/drive/MyDrive/WanStudio/models/Wan2.2-I2V-A14B')
LORA_REPO_ID = 'lkzd7/WAN2.2_LoraSet_NSFW'
LORA_MODEL_DIR = Path('/content/drive/MyDrive/WanStudio/models/WAN2.2_LoraSet_NSFW')
WEIGHT_SUFFIXES = {'.safetensors', '.bin', '.pt', '.pth', '.ckpt'}

!pip install -U "huggingface_hub[cli]"

def has_weights(model_dir):
    return model_dir.exists() and any(
        path.suffix in WEIGHT_SUFFIXES for path in model_dir.rglob('*') if path.is_file()
    )

if has_weights(FAST_BASE_MODEL_DIR):
    print('Fast base model already exists in Drive:', FAST_BASE_MODEL_DIR)
else:
    !hf download {FAST_BASE_REPO_ID} --local-dir {FAST_BASE_MODEL_DIR}

if USE_A14B_I2V_FOR_LORA_PRESET:
    if has_weights(A14B_I2V_MODEL_DIR):
        print('A14B I2V base model already exists in Drive:', A14B_I2V_MODEL_DIR)
    else:
        !hf download {A14B_I2V_REPO_ID} --local-dir {A14B_I2V_MODEL_DIR}

if has_weights(LORA_MODEL_DIR):
    print('LoRA set already exists in Drive:', LORA_MODEL_DIR)
else:
    !hf download {LORA_REPO_ID} --local-dir {LORA_MODEL_DIR}

print('Use this fast model folder in Wan Studio:', FAST_BASE_MODEL_DIR)
print('Use this A14B LoRA-compatible model folder if enabled:', A14B_I2V_MODEL_DIR)
print('Optional LoRA folder:', LORA_MODEL_DIR)


## 3. Install the official Wan runner

Wan Studio calls the official `generate.py` script from `Wan-Video/Wan2.2` when launched with `--runner wan`. If dependency installation fails around `flash_attn`, rerun the cell once; Colab package resolution can be a little fussy.


In [ ]:
WAN_REPO_DIR = '/content/Wan2.2'

!rm -rf {WAN_REPO_DIR}
!git clone https://github.com/Wan-Video/Wan2.2.git {WAN_REPO_DIR}
%cd {WAN_REPO_DIR}
!pip install -r requirements.txt


## 4. Launch the Web UI in real generation mode

Keep this cell running. Colab will show the Web UI iframe and print an `Open Wan Studio Web UI:` proxy link. Use that proxy link, not a `0.0.0.0` or `127.0.0.1` link.

In the Web UI, connect `/content/drive/MyDrive/WanStudio/models/Wan2.2-TI2V-5B` for the fast starter preset, or `/content/drive/MyDrive/WanStudio/models/Wan2.2-I2V-A14B` for the default LoRA quality preset. To attach an adapter, paste `/content/drive/MyDrive/WanStudio/models/WAN2.2_LoraSet_NSFW` into `LoRA adapter folder or file`, scan, and pick a compatible embedded preset. LOW/HIGH pairs are grouped automatically.


In [ ]:
%cd /content/wan-studio
!python wan_studio.py run --host 127.0.0.1 --port 7860 --share --runner wan --wan-repo-dir /content/Wan2.2


## Notes

- `lkzd7/WAN2.2_LoraSet_NSFW` is a LoRA/adapters set, not a standalone base model. The Web UI can attach selected `.safetensors` files as a LoRA layer.
- Real Wan2.2-TI2V-5B generation needs a 24 GB+ GPU. A14B quality presets are much larger and need stronger hardware or future optimized runners.
- Generated files are written to `/content/wan-studio/outputs` and linked in the Jobs panel.
